# Simulation for Research Note

In [12]:
"""
SU(2) Quantum Gate Estimation: Reference-Measure Geometry
==========================================================

Replication code for:
  "Reference-Measure Geometry in Quantum Parameter Estimation:
   When Coordinate Surrogates Produce Spurious Optima"
  Fulton & Fulton, submitted to Physical Review A (2026)

This script implements the multi-start gate-estimation experiment
described in Section "Computational Study".

Design
------
- True gate U(v_true) parametrised in axis-angle (exponential) coords v in R^3
- Born-rule likelihood from random pure input states measured in computational basis
- Bit-flip measurement noise with probability p_flip (default 0.01)
- Two regularised MAP objectives:

    L_E(v) = -log p(data|U(v)) + (lam/2)||v||^2            [Euclidean/flat]
    L_G(v) = -log p(data|U(v)) + (lam/2)||v||^2 - log J(v)  [Haar-consistent]

  where J(v) = [sin(||v||/2) / (||v||/2)]^2  is the Haar chart-volume factor.

- L-BFGS-B with box constraints v_i in [-3, 3]
- 5 configurations x 340 random starts = 1,700 total optimisations

Ridge-side diagnostics at each flat solution v_E*:
  (i)   ||grad L_G(v_E*)||       geometric gradient magnitude (should be ~0 if
                                  flat and geom share stationary points -- they don't)
  (ii)  Descent certificate       one step along -grad L_G strictly decreases L_G
  (iii) Intrinsic loss gap        L_G(v_E*) - L_G(v_G*)
  (iv)  Gate infidelity to truth  infid_E = 1-F(U_true, U_E*),  infid_G = 1-F(U_true, U_G*)
       + optional: difference infid_E - infid_G (kept for reference, not primary)

"""

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Any, Optional, Tuple
from scipy.optimize import minimize
import time
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================================
# SU(2) Utilities: Lie-algebra exponential map
# ============================================================================

def pauli_vector_product(v: np.ndarray) -> np.ndarray:
    """Compute v . sigma = v_x sigma_x + v_y sigma_y + v_z sigma_z."""
    sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
    sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)
    return v[0] * sigma_x + v[1] * sigma_y + v[2] * sigma_z


def axis_angle_to_su2(v: np.ndarray, eps: float = 1e-15) -> np.ndarray:
    """
    Exponential map:  v in R^3  -->  U in SU(2).

        U(v) = exp(i v.sigma/2) = cos(r/2) I + i sin(r/2) (v_hat . sigma)

    where r = ||v||.
    """
    r = np.linalg.norm(v)
    if r < eps:
        return np.eye(2, dtype=complex) + 1j * 0.5 * pauli_vector_product(v)
    v_hat = v / r
    c = np.cos(r / 2.0)
    s = np.sin(r / 2.0)
    return c * np.eye(2, dtype=complex) + 1j * s * pauli_vector_product(v_hat)


def haar_jacobian(v: np.ndarray, eps: float = 1e-15) -> float:
    """
    Haar chart-volume factor in exponential coordinates:

        J(v) = [sin(||v||/2) / (||v||/2)]^2

    Satisfies J(0) = 1 by continuity.  Near identity: J(v) ~ 1 - ||v||^2/12.
    """
    r = np.linalg.norm(v)
    if r < eps:
        return 1.0 - (r ** 2) / 12.0
    half_r = r / 2.0
    ratio = np.sin(half_r) / half_r
    return float(ratio ** 2)


def log_haar_jacobian(v: np.ndarray, eps: float = 1e-15) -> float:
    """log J(v), numerically safe near origin."""
    J = haar_jacobian(v, eps)
    return float(np.log(max(J, eps)))


def grad_log_jacobian(v: np.ndarray, eps: float = 1e-15) -> np.ndarray:
    """
    Gradient of log J(v) where J(v) = [sin(r/2)/(r/2)]^2.

        log J = 2 [log sin(r/2) - log(r/2)]
        d/dr log J = cot(r/2) - 2/r
        grad log J = [d/dr log J] * (v/r)

    Near identity:  grad log J(v) ~ -v/6 + O(||v||^3).
    """
    r = np.linalg.norm(v)
    if r < eps:
        return -v / 6.0
    half_r = r / 2.0
    sin_half = np.sin(half_r)
    cos_half = np.cos(half_r)
    if abs(sin_half) < eps:
        return np.zeros(3)
    d_logJ_dr = (cos_half / sin_half) - (2.0 / r)
    return d_logJ_dr * (v / r)


def gate_fidelity(U1: np.ndarray, U2: np.ndarray) -> float:
    """Process fidelity for single-qubit gates: F = |Tr(U1^dag U2)|^2 / 4."""
    return float(np.abs(np.trace(U1.conj().T @ U2)) ** 2 / 4.0)


def gate_infidelity(U1: np.ndarray, U2: np.ndarray) -> float:
    """1 - F."""
    return 1.0 - gate_fidelity(U1, U2)


# ============================================================================
# Quantum Measurement Simulation  (Born rule + bit-flip noise)
# ============================================================================

@dataclass
class MeasurementData:
    """Simulated measurement outcomes from a quantum gate."""
    input_states: np.ndarray   # (n_shots, 2) complex: input |psi>
    outcomes: np.ndarray       # (n_shots,) int: measurement outcome 0 or 1
    n_shots: int
    p_flip: float = 0.01      # bit-flip error probability


def simulate_gate_measurements(
    U_true: np.ndarray,
    n_shots: int,
    p_flip: float = 0.01,
    rng: Optional[np.random.Generator] = None,
) -> MeasurementData:
    """
    Simulate single-qubit gate estimation data.

    Protocol:
      1. Sample random pure input states uniformly on Bloch sphere
      2. Evolve: |psi'> = U_true |psi>
      3. Measure in computational basis (Born rule)
      4. Apply independent bit-flip error with probability p_flip

    NOTE: p_flip is a probability in [0,1], not a standard deviation.
    """
    if rng is None:
        rng = np.random.default_rng()

    input_states = np.empty((n_shots, 2), dtype=complex)
    outcomes = np.empty(n_shots, dtype=int)

    for i in range(n_shots):
        # Haar-uniform random state on Bloch sphere
        u = 2.0 * rng.random() - 1.0
        theta = np.arccos(u)
        phi = 2.0 * np.pi * rng.random()
        psi = np.array([
            np.cos(theta / 2.0),
            np.exp(1j * phi) * np.sin(theta / 2.0)
        ], dtype=complex)

        psi_out = U_true @ psi
        p1 = float(np.abs(psi_out[1]) ** 2)
        y = int(rng.random() < p1)

        if p_flip > 0.0 and (rng.random() < p_flip):
            y = 1 - y

        input_states[i, :] = psi
        outcomes[i] = y

    return MeasurementData(
        input_states=input_states, outcomes=outcomes,
        n_shots=n_shots, p_flip=p_flip,
    )


# ============================================================================
# Negative log-likelihood (Born rule) and central-difference gradient
# ============================================================================

def negative_log_likelihood(
    v: np.ndarray, data: MeasurementData, eps: float = 1e-15
) -> float:
    """
    NLL = -sum_i log p(m_i | psi_i, U(v)),
    where p(m|psi,U) = |<m|U|psi>|^2.

    This is intrinsically defined on SU(2).
    """
    U = axis_angle_to_su2(v)
    psi_out = (U @ data.input_states.T).T          # (n_shots, 2)
    p0 = np.abs(psi_out[:, 0]) ** 2
    p1 = np.abs(psi_out[:, 1]) ** 2
    p = np.where(data.outcomes == 0, p0, p1)
    p = np.clip(p, eps, 1.0)
    return float(-np.sum(np.log(p)))


def grad_nll_central(
    v: np.ndarray, data: MeasurementData,
    eps: float = 1e-15, h0: float = 1e-6,
) -> np.ndarray:
    """
    Central-difference gradient of NLL:
        d f / d v_i  ~  [f(v + h e_i) - f(v - h e_i)] / (2h)
    with mild scale adaptation:  h = h0 * max(1, |v_i|).
    """
    grad = np.zeros(3, dtype=float)
    for i in range(3):
        h = h0 * max(1.0, abs(float(v[i])))
        vp = v.copy(); vp[i] += h
        vm = v.copy(); vm[i] -= h
        grad[i] = (negative_log_likelihood(vp, data, eps)
                    - negative_log_likelihood(vm, data, eps)) / (2.0 * h)
    return grad


# ============================================================================
# Objectives:  flat (Euclidean) vs geometric (Haar-consistent)
# ============================================================================

@dataclass
class OptimizationParams:
    """Hyperparameters and data for the optimisation."""
    data: MeasurementData
    lam: float = 0.5            # regularisation strength
    eps: float = 1e-15
    fd_h0: float = 1e-6        # finite-difference step for NLL gradient
    v_bound: float = 3.0       # L-BFGS-B box constraint per component


def loss_flat(v: np.ndarray, params: OptimizationParams) -> float:
    """L_E(v) = -log p(data|U(v)) + (lam/2)||v||^2"""
    nll = negative_log_likelihood(v, params.data, params.eps)
    return nll + 0.5 * params.lam * float(np.dot(v, v))


def grad_loss_flat(v: np.ndarray, params: OptimizationParams) -> np.ndarray:
    """grad L_E = grad NLL + lam * v"""
    return grad_nll_central(v, params.data, params.eps, params.fd_h0) + params.lam * v


def loss_geom(v: np.ndarray, params: OptimizationParams) -> float:
    """L_G(v) = -log p(data|U(v)) + (lam/2)||v||^2 - log J(v)"""
    nll = negative_log_likelihood(v, params.data, params.eps)
    reg = 0.5 * params.lam * float(np.dot(v, v))
    logJ = log_haar_jacobian(v, params.eps)
    return nll + reg - logJ


def grad_loss_geom(v: np.ndarray, params: OptimizationParams) -> np.ndarray:
    """grad L_G = grad NLL + lam * v - grad log J(v)"""
    return (grad_nll_central(v, params.data, params.eps, params.fd_h0)
            + params.lam * v
            - grad_log_jacobian(v, params.eps))


# ============================================================================
# Multi-start L-BFGS-B optimiser
# ============================================================================

def optimize_from_start(
    v0: np.ndarray,
    which: str,
    params: OptimizationParams,
    method: str = "L-BFGS-B",
) -> Dict[str, Any]:
    """Run L-BFGS-B from v0 on the flat or geometric objective."""
    if which == "flat":
        fun = lambda v: loss_flat(v, params)
        jac = lambda v: grad_loss_flat(v, params)
    elif which == "geom":
        fun = lambda v: loss_geom(v, params)
        jac = lambda v: grad_loss_geom(v, params)
    else:
        raise ValueError("which must be 'flat' or 'geom'")

    b = params.v_bound
    res = minimize(
        fun=fun, x0=v0, jac=jac, method=method,
        bounds=[(-b, b)] * 3,
        options=dict(maxiter=800, ftol=1e-12, gtol=1e-10),
    )
    v_star = np.array(res.x, dtype=float)
    return dict(
        which=which,
        success=bool(res.success),
        v_star=v_star,
        norm_v_star=float(np.linalg.norm(v_star)),
        L_flat=float(loss_flat(v_star, params)),
        L_geom=float(loss_geom(v_star, params)),
        grad_flat_norm=float(np.linalg.norm(grad_loss_flat(v_star, params))),
        grad_geom_norm=float(np.linalg.norm(grad_loss_geom(v_star, params))),
    )


# ============================================================================
# Descent certificate
# ============================================================================

def descent_certificate(
    v: np.ndarray,
    params: OptimizationParams,
    eta0: float = 1e-3,
) -> Dict[str, float]:
    """
    At v (typically v_E*), step along -grad L_G and verify L_G decreases.
    Step size scaled by 1/(1+||grad||) for conservatism.
    Returns dL < 0 if descent direction exists.
    """
    g = grad_loss_geom(v, params)
    gnorm = float(np.linalg.norm(g))
    if gnorm < 1e-14:
        return dict(eta=0.0, dL=0.0, gnorm=gnorm)
    eta = eta0 / (1.0 + gnorm)
    L0 = loss_geom(v, params)
    v1 = np.clip(v - eta * g, -params.v_bound, params.v_bound)
    L1 = loss_geom(v1, params)
    return dict(eta=float(eta), dL=float(L1 - L0), gnorm=gnorm)


# ============================================================================
# Single-configuration experiment runner
# ============================================================================

def run_single_config(
    config_name: str,
    v_true: np.ndarray,
    n_shots: int,
    lam: float,
    n_starts: int = 340,
    p_flip: float = 0.01,
    v0_scale: float = 0.3,
    v_bound: float = 3.0,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Run multi-start optimisation for one experimental configuration.

    For each random start:
      1. Optimise L_E --> v_E*   (flat/Euclidean)
      2. Optimise L_G --> v_G*   (Haar-consistent/geometric)
      3. Compute ridge-side diagnostics at v_E*
    """
    rng = np.random.default_rng(seed)
    U_true = axis_angle_to_su2(v_true)

    print(f"\n{'=' * 70}")
    print(f"CONFIG: {config_name}")
    print(f"  v_true = [{v_true[0]:.2f}, {v_true[1]:.2f}, {v_true[2]:.2f}]"
          f"  ||v_true|| = {np.linalg.norm(v_true):.4f}")
    print(f"  N = {n_shots},  lam = {lam},  p_flip = {p_flip}")
    print(f"  starts = {n_starts},  v_bound = {v_bound},  seed = {seed}")
    print(f"{'=' * 70}")

    # Generate measurement data
    data = simulate_gate_measurements(U_true, n_shots, p_flip, rng)
    params = OptimizationParams(data=data, lam=lam, v_bound=v_bound)

    # Random starting points ~ N(0, v0_scale^2 I)
    starts = rng.normal(0.0, v0_scale, size=(n_starts, 3))

    rows = []
    t0 = time.time()

    for i, v0 in enumerate(starts):
        if (i + 1) % 50 == 0 or (i + 1) == n_starts:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            eta_est = (n_starts - i - 1) / rate if rate > 0 else 0
            print(f"  [{config_name}] {i+1:>4d}/{n_starts}"
                  f"  ({100*(i+1)/n_starts:5.1f}%)"
                  f"  [{rate:.1f} iter/s, ETA {eta_est:.0f}s]")

        # Optimise both objectives from same start
        res_E = optimize_from_start(v0, "flat", params)
        res_G = optimize_from_start(v0, "geom", params)

        v_E = res_E["v_star"]
        v_G = res_G["v_star"]

        # Ridge-side diagnostics at v_E*
        grad_G_at_E = grad_loss_geom(v_E, params)
        grad_E_at_E = grad_loss_flat(v_E, params)
        cert = descent_certificate(v_E, params, eta0=1e-3)

        # Gate infidelities
        U_E = axis_angle_to_su2(v_E)
        U_G = axis_angle_to_su2(v_G)
        infid_E = gate_infidelity(U_true, U_E)
        infid_G = gate_infidelity(U_true, U_G)

        # Intrinsic loss gap: L_G(v_E*) - L_G(v_G*)
        L_G_at_E = loss_geom(v_E, params)
        L_G_at_G = res_G["L_geom"]
        geom_loss_gap = L_G_at_E - L_G_at_G

        rows.append(dict(
            config=config_name,
            start_id=i,

            # v_true info
            v_true_x=float(v_true[0]),
            v_true_y=float(v_true[1]),
            v_true_z=float(v_true[2]),
            n_shots=n_shots,
            lam=lam,
            p_flip=p_flip,
            seed=seed,

            # Flat solution
            flat_vx=float(v_E[0]),
            flat_vy=float(v_E[1]),
            flat_vz=float(v_E[2]),
            flat_norm=float(np.linalg.norm(v_E)),
            flat_L_E=float(res_E["L_flat"]),
            flat_L_G=float(L_G_at_E),
            flat_grad_E_norm=float(np.linalg.norm(grad_E_at_E)),
            flat_grad_G_norm=float(np.linalg.norm(grad_G_at_E)),
            flat_success=bool(res_E["success"]),
            flat_infidelity=infid_E,

            # Geometric solution
            geom_vx=float(v_G[0]),
            geom_vy=float(v_G[1]),
            geom_vz=float(v_G[2]),
            geom_norm=float(np.linalg.norm(v_G)),
            geom_L_E=float(res_G["L_flat"]),
            geom_L_G=float(L_G_at_G),
            geom_grad_G_norm=float(res_G["grad_geom_norm"]),
            geom_success=bool(res_G["success"]),
            geom_infidelity=infid_G,

            # Diagnostics
            geom_loss_gap=geom_loss_gap,
            cert_eta=float(cert["eta"]),
            cert_dL=float(cert["dL"]),
            cert_gnorm=float(cert["gnorm"]),
            infidelity_diff=infid_E - infid_G,

            # Distances
            distance_E_to_G=float(np.linalg.norm(v_E - v_G)),
            dist_E_to_true=float(np.linalg.norm(v_E - v_true)),
            dist_G_to_true=float(np.linalg.norm(v_G - v_true)),

            # Boundary flag
            at_boundary=bool(np.any(np.abs(v_E) > v_bound - 0.01)),
        ))

    elapsed = time.time() - t0
    print(f"  [{config_name}] Done in {elapsed:.1f}s"
          f"  ({n_starts / elapsed:.1f} iter/s)")
    return pd.DataFrame(rows)


# ============================================================================
# Full 1,700-run study  (PRA Table 2)
# ============================================================================

def run_full_study() -> pd.DataFrame:
    """
    Run all 5 configurations from PRA Table 2.
    Each has 340 starts  -->  5 x 340 = 1,700 total optimisations.

    Table 2:
      C1: v_true = ( 0.5,  0.3, -0.4),  N=1000, lam=0.01
      C2: v_true = ( 1.2, -0.8,  0.6),  N= 500, lam=0.10
      C3: v_true = ( 0.3,  1.4, -1.1),  N= 100, lam=1.00
      C4: v_true = (-0.9,  0.7,  1.3),  N= 300, lam=0.10
      C5: v_true = ( 1.5, -1.2,  0.9),  N= 200, lam=0.50

    All with p_flip=0.01, v0 ~ N(0, 0.09 I), v_bound=3.0.
    """
    configs = [

    # ============================================================
    # BASELINE — Original interior truths, lambda sweep
    # Purpose: Does Haar matter as lambda → 0 ?
    # ============================================================

    ("C0_lam0",     np.array([ 0.1,  0.2, -0.4]), 1000, 0.0,   101),
    ("C0_lam1e-6",  np.array([ 0.1,  0.2, -0.4]), 1000, 1e-6,  101),
    ("C0_lam1e-4",  np.array([ 0.1,  0.2, -0.4]), 1000, 1e-4,  101),
    ("C0_lam1e-2",  np.array([ 0.1,  0.2, -0.4]), 1000, 1e-2,  101),

    ("C1_lam0",     np.array([ 0.5,  0.3, -0.4]), 1000, 0.0,   102),
    ("C1_lam1e-6",  np.array([ 0.5,  0.3, -0.4]), 1000, 1e-6,  102),
    ("C1_lam1e-4",  np.array([ 0.5,  0.3, -0.4]), 1000, 1e-4,  102),
    ("C1_lam1e-2",  np.array([ 0.5,  0.3, -0.4]), 1000, 1e-2,  102),

    ("C2_lam0",     np.array([ 1.2, -0.8,  0.6]),  500, 0.0,   103),
    ("C2_lam1e-6",  np.array([ 1.2, -0.8,  0.6]),  500, 1e-6,  103),
    ("C2_lam1e-4",  np.array([ 1.2, -0.8,  0.6]),  500, 1e-4,  103),
    ("C2_lam1e-2",  np.array([ 1.2, -0.8,  0.6]),  500, 1e-2,  103),

    # ============================================================
    # INTERIOR PRIOR-SENSITIVE REGIME — Low data
    # Purpose: Let prior geometry influence the solution
    # ============================================================

    ("C1_lo_lam1e-2", np.array([ 0.5,  0.3, -0.4]),  50, 1e-2, 201),
    ("C1_lo_lam1e-6", np.array([ 0.5,  0.3, -0.4]),  50, 1e-6, 201),

    ("C2_lo_lam1e-2", np.array([ 1.2, -0.8,  0.6]),  50, 1e-2, 202),
    ("C2_lo_lam1e-6", np.array([ 1.2, -0.8,  0.6]),  50, 1e-6, 202),

    ("C4_lo_lam1e-2", np.array([-0.9,  0.7,  1.3]),  50, 1e-2, 203),
    ("C4_lo_lam1e-6", np.array([-0.9,  0.7,  1.3]),  50, 1e-6, 203),

    # ============================================================
    # BOUNDARY-STRESS REGIME — Large norms
    # Purpose: Force constraint interaction (when box is on)
    # ============================================================

    ("B1_small", np.array([ 2.6,  0.2, -0.1]), 100, 1e-6, 301),
    ("B2_small", np.array([ 0.1,  2.6, -0.2]), 100, 1e-6, 302),
    ("B3_small", np.array([ 1.8, -1.8,  1.2]), 100, 1e-6, 303),

    ("B1_med",   np.array([ 2.6,  0.2, -0.1]), 100, 1e-2, 311),
    ("B2_med",   np.array([ 0.1,  2.6, -0.2]), 100, 1e-2, 312),
    ("B3_med",   np.array([ 1.8, -1.8,  1.2]), 100, 1e-2, 313),

    # ============================================================
    # HIGH-DATA CONTROLS — Likelihood dominates
    # Purpose: Show geometry becomes irrelevant with enough data
    # ============================================================

    ("C3_hi", np.array([ 0.3,  1.4, -1.1]), 2000, 1e-2, 401),
    ("C4_hi", np.array([-0.9,  0.7,  1.3]), 2000, 1e-2, 402),
    ("C5_hi", np.array([ 1.5, -1.2,  0.9]), 2000, 1e-2, 403),
    ]


    n_starts_per = 340
    p_flip = 0.01
    v0_scale = 0.3       # so v0 ~ N(0, 0.09 I)
    v_bound = 3.0

    total = len(configs) * n_starts_per
    print("\n" + "=" * 70)
    print(f"SU(2) REFERENCE-MEASURE GEOMETRY STUDY  (PRA Table 2)")
    print(f"  Configurations: {len(configs)}")
    print(f"  Starts/config:  {n_starts_per}")
    print(f"  Total runs:     {total}")
    print(f"  p_flip = {p_flip},  v0 ~ N(0, {v0_scale**2:.2f} I),  v_bound = {v_bound}")
    print("=" * 70)

    all_dfs = []
    t_total = time.time()

    for name, v_true, N, lam, seed in configs:
        df = run_single_config(
            config_name=name,
            v_true=v_true,
            n_shots=N,
            lam=lam,
            n_starts=n_starts_per,
            p_flip=p_flip,
            v0_scale=v0_scale,
            v_bound=v_bound,
            seed=seed,
        )
        all_dfs.append(df)

    df_all = pd.concat(all_dfs, ignore_index=True)
    total_time = time.time() - t_total
    print(f"\nAll configs done.  Total wall time: {total_time:.0f}s"
          f" ({total_time / 60:.1f} min)")
    return df_all


# ============================================================================
# Summary statistics
# ============================================================================

def print_summary(df: pd.DataFrame):
    """Print the paper's key diagnostic statistics."""
    n = len(df)
    eps_stat = 1e-6

    print("\n" + "=" * 70)
    print(f"RESULTS SUMMARY  ({n} runs)")
    print("=" * 70)

    # ---- Global diagnostics ----
    nonstat = (df["flat_grad_G_norm"] > eps_stat).sum()
    descent_ok = (df["cert_dL"] < 0).sum()
    boundary = df["at_boundary"].sum()

    print(f"\nNon-stationary for L_G  (||grad L_G(v_E*)|| > {eps_stat}):"
          f"  {nonstat}/{n}  ({100*nonstat/n:.1f}%)")
    print(f"Descent certificates    (dL < 0):"
          f"  {descent_ok}/{n}  ({100*descent_ok/n:.1f}%)")

    print(f"\n||grad L_G(v_E*)|| distribution:")
    for q, label in [(0.25, "Q25"), (0.50, "Median"), (0.75, "Q75"),
                      (0.95, "Q95"), (0.99, "Q99")]:
        print(f"  {label:>6s}: {df['flat_grad_G_norm'].quantile(q):.4f}")
    print(f"  {'Max':>6s}: {df['flat_grad_G_norm'].max():.4f}")

    print(f"\nMedian ||v_E*||: {df['flat_norm'].median():.4f}")
    print(f"Median ||v_G*||: {df['geom_norm'].median():.4f}")

    print(f"\nBoundary cases: {boundary}/{n} ({100*boundary/n:.1f}%)")
    if boundary > 0:
        bd = df[df["at_boundary"]]
        print(f"  Max ||grad L_G|| at boundary: {bd['flat_grad_G_norm'].max():.1f}")

        # ---- Infidelity to truth (primary performance metric) ----
    print(f"\nInfidelity to truth (1 - F(U_true, U_*)):")
    for col, name in [("flat_infidelity", "Flat (v_E*)"),
                      ("geom_infidelity", "Geom (v_G*)")]:
        print(f"  {name}:")
        print(f"    Median: {df[col].median():.6f}")
        print(f"    IQR:    [{df[col].quantile(0.25):.6f}, {df[col].quantile(0.75):.6f}]")
        print(f"    P95:    {df[col].quantile(0.95):.6f}")
        print(f"    Max:    {df[col].max():.6f}")

    # Boundary-only performance (what you actually care about for the ridge narrative)
    if boundary > 0:
        bd = df[df["at_boundary"]]
        print(f"\nBoundary-only infidelity to truth (n={len(bd)}):")
        for col, name in [("flat_infidelity", "Flat (v_E*)"),
                          ("geom_infidelity", "Geom (v_G*)")]:
            print(f"  {name}: median={bd[col].median():.6f}, "
                  f"IQR=[{bd[col].quantile(0.25):.6f}, {bd[col].quantile(0.75):.6f}], "
                  f"max={bd[col].max():.6f}")

    # Keep the diff metric as a *secondary* quantity (useful but not the headline)
    print(f"\nSecondary: Infidelity difference (flat minus geom):")
    print(f"  Median: {df['infidelity_diff'].median():.6f}")
    print(f"  IQR:    [{df['infidelity_diff'].quantile(0.25):.6f}, {df['infidelity_diff'].quantile(0.75):.6f}]")
    print(f"  Max:    {df['infidelity_diff'].max():.6f}")


    # ---- Per-configuration breakdown ----
    print(f"\n{'=' * 70}")
    print("PER-CONFIGURATION BREAKDOWN")
    print(f"{'=' * 70}")
    print(f"{'Config':<8s} {'N':>5s} {'lam':>6s} {'||vt||':>7s}"
          f" {'med||gG||':>10s} {'bdy':>4s}"
          f" {'med_infid_E':>12s} {'med_infid_G':>12s}"
          f" {'bdy_med_E':>11s} {'bdy_med_G':>11s}"
          f" {'descent':>8s}")
    print("-" * 90)
    
    for config in df["config"].unique():
    
        dc = df[df["config"] == config]
    
        vt = np.array([
            dc["v_true_x"].iloc[0],
            dc["v_true_y"].iloc[0],
            dc["v_true_z"].iloc[0]
        ])
    
        # Boundary subset for this config
        bd_c = dc[dc["at_boundary"]]
    
        bdy_med_E = bd_c["flat_infidelity"].median() if len(bd_c) > 0 else np.nan
        bdy_med_G = bd_c["geom_infidelity"].median() if len(bd_c) > 0 else np.nan
    
        print(f"{config:<8s}"
              f" {dc['n_shots'].iloc[0]:>5d}"
              f" {dc['lam'].iloc[0]:>6.2f}"
              f" {np.linalg.norm(vt):>7.3f}"
              f" {dc['flat_grad_G_norm'].median():>10.4f}"
              f" {dc['at_boundary'].sum():>4d}"
              f" {dc['flat_infidelity'].median():>12.6f}"
              f" {dc['geom_infidelity'].median():>12.6f}"
              f" {bdy_med_E:>11.6f}"
              f" {bdy_med_G:>11.6f}"
              f" {(dc['cert_dL'] < 0).sum():>4d}/{len(dc)}")




# ============================================================================
# Publication-quality diagnostic plots
# ============================================================================

def make_diagnostic_plots(
    df: pd.DataFrame, out_png: str = "su2_pra_diagnostics.png"
):
    """Generate the 4-panel diagnostic figure."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))

    # ---- Panel A: ||grad L_G(v_E*)|| histogram ----
    ax = axes[0, 0]
    ax.hist(df["flat_grad_G_norm"], bins=60,
            color="steelblue", edgecolor="black", alpha=0.8)
    med = df["flat_grad_G_norm"].median()
    ax.axvline(med, color="red", linestyle="--", linewidth=2,
               label=f"Median = {med:.3f}")
    ax.set_xlabel(r"$\|\nabla L_G(v_E^*)\|$", fontsize=13)
    ax.set_ylabel("Count", fontsize=12)
    ax.set_title("(A)  Geometric gradient at flat solutions", fontsize=13,
                 fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

    # ---- Panel B: ||v_E*|| vs ||v_G*|| ----
    ax = axes[0, 1]
    ax.scatter(df["flat_norm"], df["geom_norm"],
               alpha=0.25, s=10, c="steelblue")
    maxval = max(df["flat_norm"].max(), df["geom_norm"].max()) * 1.05
    ax.plot([0, maxval], [0, maxval], "r--", linewidth=1.5, label="y = x")
    ax.set_xlabel(r"$\|v_E^*\|$", fontsize=13)
    ax.set_ylabel(r"$\|v_G^*\|$", fontsize=13)
    ax.set_title("(B)  Flat vs geometric solution norms", fontsize=13,
                 fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

    # ---- Panel C: Infidelity difference by config ----
    # ---- Panel C: Boundary-only infidelity to truth by config ----
    ax = axes[1, 0]
    configs = df["config"].unique()

    bd = df[df["at_boundary"]]
    if len(bd) == 0:
        ax.text(0.5, 0.5, "No boundary cases in this run",
                ha="center", va="center", fontsize=12)
        ax.axis("off")
    else:
        data_E = [bd[bd["config"] == c]["flat_infidelity"].values for c in configs]
        data_G = [bd[bd["config"] == c]["geom_infidelity"].values for c in configs]

        pos_E = np.arange(len(configs)) * 2.0
        pos_G = pos_E + 0.7

        bpE = ax.boxplot(data_E, positions=pos_E, widths=0.6, patch_artist=True)
        bpG = ax.boxplot(data_G, positions=pos_G, widths=0.6, patch_artist=True)

        for patch in bpE["boxes"]:
            patch.set_facecolor("steelblue"); patch.set_alpha(0.6)
        for patch in bpG["boxes"]:
            patch.set_facecolor("orange"); patch.set_alpha(0.6)

        ax.set_xticks(pos_E + 0.35)
        ax.set_xticklabels(configs)
        ax.set_xlabel("Configuration", fontsize=12)
        ax.set_ylabel(r"Boundary infidelity to truth  $(1-F(U_{\mathrm{true}}, U_*))$", fontsize=12)
        ax.set_title("(C)  Boundary-only infidelity to truth", fontsize=13, fontweight="bold")
        ax.grid(alpha=0.3, axis="y")
        ax.legend([bpE["boxes"][0], bpG["boxes"][0]], ["Flat (v_E*)", "Geom (v_G*)"],
                  fontsize=10, loc="upper left")


    # ---- Panel D: ||grad L_G(v_E*)|| vs ||v_E*|| ----
    ax = axes[1, 1]
    ax.scatter(df["flat_norm"], df["flat_grad_G_norm"],
               alpha=0.25, s=10, c="steelblue")
    ax.set_xlabel(r"$\|v_E^*\|$", fontsize=13)
    ax.set_ylabel(r"$\|\nabla L_G(v_E^*)\|$", fontsize=13)
    ax.set_title("(D)  Gradient magnitude vs solution norm",
                 fontsize=13, fontweight="bold")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"\nDiagnostic plot saved: {out_png}")


# ============================================================================
# Top-10 extreme ridge cases  (for paper narrative)
# ============================================================================

def print_extreme_cases(df: pd.DataFrame, n: int = 10):
    """Print the most extreme ridge cases for paper discussion."""
    print(f"\n{'=' * 70}")
    print(f"TOP {n} EXTREME RIDGE CASES")
    print(f"{'=' * 70}")
    top = df.nlargest(n, "flat_grad_G_norm")
    for rank, (idx, row) in enumerate(top.iterrows(), 1):
        print(f"\nRank {rank}:  Config {row['config']}")
        print(f"  v_E* = [{row['flat_vx']:.4f}, {row['flat_vy']:.4f},"
              f" {row['flat_vz']:.4f}]   ||v_E*|| = {row['flat_norm']:.4f}")
        print(f"  ||grad L_G(v_E*)|| = {row['flat_grad_G_norm']:.4f}")
        print(f"  Descent cert dL    = {row['cert_dL']:.6f}")
        print(f"  Infidelity to truth (flat) = {row['flat_infidelity']:.6f}")
        print(f"  Infidelity to truth (geom) = {row['geom_infidelity']:.6f}")
        print(f"  Infidelity diff (E-G)      = {row['infidelity_diff']:.6f}")

        print(f"  At boundary:         {row['at_boundary']}")


# ============================================================================
# Main
# ============================================================================

if __name__ == "__main__":

    # --- Run the full 1,700-run study (PRA Table 2) ---
    df = run_full_study()

    # --- Save results ---
    csv_path = "su2_pra_table2_results_1700.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nResults saved: {csv_path}")

    # --- Summary ---
    print_summary(df)

    # --- Extreme cases ---
    print_extreme_cases(df)

    # --- Diagnostic plots ---
    make_diagnostic_plots(df, out_png="su2_pra_diagnostics.png")



SU(2) REFERENCE-MEASURE GEOMETRY STUDY  (PRA Table 2)
  Configurations: 27
  Starts/config:  340
  Total runs:     9180
  p_flip = 0.01,  v0 ~ N(0, 0.09 I),  v_bound = 3.0

CONFIG: C0_lam0
  v_true = [0.10, 0.20, -0.40]  ||v_true|| = 0.4583
  N = 1000,  lam = 0.0,  p_flip = 0.01
  starts = 340,  v_bound = 3.0,  seed = 101
  [C0_lam0]   50/340  ( 14.7%)  [86.4 iter/s, ETA 3s]
  [C0_lam0]  100/340  ( 29.4%)  [88.7 iter/s, ETA 3s]
  [C0_lam0]  150/340  ( 44.1%)  [61.0 iter/s, ETA 3s]
  [C0_lam0]  200/340  ( 58.8%)  [65.5 iter/s, ETA 2s]
  [C0_lam0]  250/340  ( 73.5%)  [69.5 iter/s, ETA 1s]
  [C0_lam0]  300/340  ( 88.2%)  [72.1 iter/s, ETA 1s]
  [C0_lam0]  340/340  (100.0%)  [73.4 iter/s, ETA 0s]
  [C0_lam0] Done in 4.6s  (73.2 iter/s)

CONFIG: C0_lam1e-6
  v_true = [0.10, 0.20, -0.40]  ||v_true|| = 0.4583
  N = 1000,  lam = 1e-06,  p_flip = 0.01
  starts = 340,  v_bound = 3.0,  seed = 101
  [C0_lam1e-6]   50/340  ( 14.7%)  [93.0 iter/s, ETA 3s]
  [C0_lam1e-6]  100/340  ( 29.4%)  [96.0 it

In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

def make_extreme_tables(
    df: pd.DataFrame,
    n: int = 10,
    grad_col: str = "flat_grad_G_norm",
    boundary_col: str = "at_boundary",
    config_col: str = "config",
    tie_breakers: tuple = ("cert_dL", "flat_infidelity", "start_id"),
    sort_desc: bool = True,
    verbose: bool = True,
    round_cols: dict | None = None,
) -> dict[str, pd.DataFrame]:
    """
    Produce 'extreme ridge' tables split by boundary flag.

    Key upgrades vs original:
      - stable sorting with explicit tie-breakers
      - optional rounding for paper tables
      - no side effects (printing is optional)
    """
    required = {grad_col, boundary_col, config_col}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Coerce boundary flag robustly
    bd = df[boundary_col].astype(bool)

    df_interior = df.loc[~bd].copy()
    df_boundary = df.loc[bd].copy()

    # violation_score = grad * max(gap, 0)  — joint severity metric
    for _frame in (df_interior, df_boundary):
        if "geom_loss_gap" in _frame.columns:
            _frame["violation_score"] = (
                _frame[grad_col] * _frame["geom_loss_gap"].clip(lower=0)
            )
        else:
            _frame["violation_score"] = _frame[grad_col]

    # Columns worth showing (keep only those that exist)
    preferred_cols = [
        config_col, "start_id",
        "flat_vx", "flat_vy", "flat_vz", "flat_norm",
        "violation_score", grad_col, "cert_dL",
        "flat_infidelity", "geom_infidelity", "infidelity_diff",
        "geom_loss_gap",
        "distance_E_to_G", "dist_E_to_true", "dist_G_to_true",
        boundary_col,
    ]
    show_cols = [c for c in preferred_cols if c in df.columns]

    # Build sort keys: violation_score primary, grad secondary, then tie-breakers
    _score_col = "violation_score" if "violation_score" in df_interior.columns else grad_col
    sort_keys = [_score_col, grad_col] + [c for c in tie_breakers if c in df.columns]
    ascending = [not sort_desc, not sort_desc] + [True] * (len(sort_keys) - 2)

    def topk(d: pd.DataFrame) -> pd.DataFrame:
        if len(d) == 0:
            return d[show_cols].copy()
        # stable sort makes results deterministic even with ties
        out = (
            d.sort_values(sort_keys, ascending=ascending, kind="mergesort")
             .head(n)[show_cols]
             .reset_index(drop=True)
        )
        return out

    interior_top = topk(df_interior)
    boundary_top = topk(df_boundary)

    # Per-config summary: counts + max/median grad in each region
    rows = []
    # Keep config order consistent with appearance in df (not alphabetical)
    cfg_order = pd.unique(df[config_col])
    for cfg in cfg_order:
        dcfg = df[df[config_col] == cfg]
        bcfg = dcfg[dcfg[boundary_col].astype(bool)]
        icfg = dcfg[~dcfg[boundary_col].astype(bool)]
        rows.append({
            "config": cfg,
            "n_total": len(dcfg),
            "n_boundary": len(bcfg),
            "n_interior": len(icfg),
            "max_grad_interior": float(icfg[grad_col].max()) if len(icfg) else np.nan,
            "max_grad_boundary": float(bcfg[grad_col].max()) if len(bcfg) else np.nan,
            "median_grad_interior": float(icfg[grad_col].median()) if len(icfg) else np.nan,
            "median_grad_boundary": float(bcfg[grad_col].median()) if len(bcfg) else np.nan,
        })
    per_config = (
        pd.DataFrame(rows)
          .sort_values(["n_boundary", "max_grad_boundary"], ascending=[False, False], kind="mergesort")
          .reset_index(drop=True)
    )

    # Optional rounding for paper display
    if round_cols:
        for col, nd in round_cols.items():
            if col in interior_top.columns:
                interior_top[col] = interior_top[col].round(nd)
            if col in boundary_top.columns:
                boundary_top[col] = boundary_top[col].round(nd)
            if col in per_config.columns:
                per_config[col] = per_config[col].round(nd)

    if verbose:
        print("\n" + "=" * 86)
        print(f"TOP {n} INTERIOR EXTREMES  ({boundary_col} == False)  n={len(df_interior)}")
        print("=" * 86)
        print(interior_top.to_string(index=False) if len(df_interior) else "No interior points.")

        print("\n" + "=" * 86)
        print(f"TOP {n} BOUNDARY EXTREMES  ({boundary_col} == True)   n={len(df_boundary)}")
        print("=" * 86)
        print(boundary_top.to_string(index=False) if len(df_boundary) else "No boundary points.")

        print("\n" + "=" * 86)
        print("PER-CONFIG EXTREME SUMMARY")
        print("=" * 86)
        print(per_config.to_string(index=False))

    return {
        "interior_top": interior_top,
        "boundary_top": boundary_top,
        "per_config_summary": per_config,
    }


def save_extreme_tables(
    tables: dict[str, pd.DataFrame],
    out_dir: str | Path = ".",
    stem: str = "extreme_ridge",
    also_latex: bool = True,
) -> dict[str, Path]:
    """
    Save CSVs (and optionally LaTeX) with timestamped filenames.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    paths = {}
    for key, df in tables.items():
        csv_path = out_dir / f"{stem}_{key}_{ts}.csv"
        df.to_csv(csv_path, index=False)
        paths[key] = csv_path

        if also_latex:
            tex_path = out_dir / f"{stem}_{key}_{ts}.tex"
            # minimal, drop-in LaTeX table
            df.to_latex(tex_path, index=False, escape=False)
            paths[f"{key}_latex"] = tex_path

    return paths


# --- usage (kept explicit; no side effects on import) ---
df = pd.read_csv("su2_pra_table2_results_1700.csv")

tables = make_extreme_tables(
    df,
    n=10,
    verbose=True,
    round_cols={
        "flat_vx": 4, "flat_vy": 4, "flat_vz": 4,
        "flat_norm": 4,
        "flat_grad_G_norm": 4,
        "cert_dL": 6,
        "flat_infidelity": 6, "geom_infidelity": 6, "infidelity_diff": 6,
    }
)

paths = save_extreme_tables(tables, out_dir=".", stem="su2_pra_top10", also_latex=True)
print("\nSaved:")
for k, p in paths.items():
    print(f"  {k:>20s}: {p}")



TOP 10 INTERIOR EXTREMES  (at_boundary == False)  n=9041
       config  start_id  flat_vx  flat_vy  flat_vz  flat_norm  flat_grad_G_norm   cert_dL  flat_infidelity  geom_infidelity  infidelity_diff  geom_loss_gap  distance_E_to_G  dist_E_to_true  dist_G_to_true  at_boundary
     B1_small       102   0.8477   2.0558  -2.0636     3.0337           34.6327 -0.033642         0.872165         0.024052         0.848113      24.434141         3.523874        3.220292        0.420025        False
   C2_lam1e-6       146  -2.5657   2.8773  -2.3938     4.5378            1.2798 -0.000689         0.049068         0.111735        -0.062668       2.001489         5.642704        6.055236        0.742560        False
      C2_lam0       146  -2.5657   2.8773  -2.3938     4.5378            1.2798 -0.000689         0.049068         0.111735        -0.062667       2.001480         5.642704        6.055236        0.742560        False
C2_lo_lam1e-6       128  -2.5639   2.8909  -1.9119     4.3112         